In [ ]:
import pandas as pd


df = pd.read_json("data/game_stats.json")
df

<>:4: SyntaxWarning: "\g" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\g"? A raw string is also an option.
<>:4: SyntaxWarning: "\g" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\g"? A raw string is also an option.
C:\Users\nihaa\AppData\Local\Temp\ipykernel_27424\2472310309.py:4: SyntaxWarning: "\g" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\g"? A raw string is also an option.
  df=pd.read_json('Files\game_stats.json')


,stat_id,game_id,player_id,team_id,goals,assists,points,shots_on_goal,penalty_min,toi,plus_minus
0,1,2025010051,8480448,1,0,0,0,0,0,14:28,0
1,2,2025010051,8479525,1,0,0,0,0,0,18:48,-1
2,3,2025010051,8477492,1,0,1,1,0,0,18:57,1
3,6,2025010051,8477476,1,1,1,2,0,0,20:44,2
4,10,2025010051,8482947,1,1,0,1,0,0,12:48,1
...,...,...,...,...,...,...,...,...,...,...,...
47403,53923,2025021300,8476879,20,0,0,0,0,0,18:25,-1
47404,53924,2025021300,8476441,20,0,0,0,0,0,18:48,0
47405,53925,2025021300,8474563,20,0,1,1,0,0,22:20,1
47406,53926,2025021300,8479998,20,0,0,0,0,0,19:50,2


In [2]:
import requests
import pandas as pd
import pymysql

conn = pymysql.connect(
    host="localhost",
    user="root",
    password="root",
    database="nhl",
)

try:
    team_abbrevs = pd.read_sql(
        "SELECT team_abbrev FROM teams ORDER BY team_abbrev",
        conn,
    )["team_abbrev"].dropna().tolist()
finally:
    conn.close()

game_records = []

for team_abbrev in team_abbrevs:
    url = f"https://api-web.nhle.com/v1/club-schedule-season/{team_abbrev}/20252026"
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        schedule = response.json()
    except requests.RequestException as error:
        print(f"Could not load schedule for {team_abbrev}: {error}")
        continue

    for game in schedule.get("games", []):
        home_team = game.get("homeTeam", {})
        away_team = game.get("awayTeam", {})
        game_records.append({
            "game_id": game.get("id"),
            "season": game.get("season"),
            "game_type": game.get("gameType"),
            "game_date": game.get("gameDate"),
            "start_time_utc": game.get("startTimeUTC"),
            "game_state": game.get("gameState"),
            "home_team_abbrev": home_team.get("abbrev"),
            "home_team_name": home_team.get("commonName", {}).get("default", ""),
            "home_score": home_team.get("score"),
            "away_team_abbrev": away_team.get("abbrev"),
            "away_team_name": away_team.get("commonName", {}).get("default", ""),
            "away_score": away_team.get("score"),
            "venue": game.get("venue", {}).get("default", ""),
        })

schedule_df = pd.DataFrame(game_records).drop_duplicates("game_id")
schedule_df = schedule_df.sort_values("game_date").reset_index(drop=True)
schedule_df

C:\Users\nihaa\AppData\Local\Temp\ipykernel_10988\1380257707.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  team_abbrevs = pd.read_sql(


,game_id,season,game_type,game_date,start_time_utc,game_state,home_team_abbrev,home_team_name,home_score,away_team_abbrev,away_team_name,away_score,venue
0,2025010101,20252026,1,2025-09-20,2025-09-20T23:00:00Z,FINAL,DAL,Stars,2,STL,Blues,1,American Airlines Center
1,2025010001,20252026,1,2025-09-21,2025-09-21T22:00:00Z,FINAL,LAK,Kings,3,ANA,Ducks,1,Toyota Arena
2,2025010003,20252026,1,2025-09-21,2025-09-22T00:00:00Z,FINAL,CGY,Flames,0,EDM,Oilers,3,Scotiabank Saddledome
3,2025010002,20252026,1,2025-09-21,2025-09-22T00:00:00Z,FINAL,EDM,Oilers,2,CGY,Flames,3,Rogers Place
4,2025010004,20252026,1,2025-09-21,2025-09-21T19:00:00Z,FINAL,NSH,Predators,5,FLA,Panthers,0,Bridgestone Arena
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1493,2025030412,20252026,3,2026-06-04,2026-06-05T00:00:00Z,OFF,CAR,Hurricanes,4,VGK,Golden Knights,3,Lenovo Center
1494,2025030413,20252026,3,2026-06-06,2026-06-07T00:00:00Z,OFF,VGK,Golden Knights,5,CAR,Hurricanes,4,T-Mobile Arena
1495,2025030414,20252026,3,2026-06-09,2026-06-10T00:00:00Z,OFF,VGK,Golden Knights,3,CAR,Hurricanes,5,T-Mobile Arena
1496,2025030415,20252026,3,2026-06-11,2026-06-12T00:00:00Z,OFF,CAR,Hurricanes,4,VGK,Golden Knights,2,Lenovo Center


In [6]:

import pymysql

conn = pymysql.connect(
    host='localhost',
    user='root',
    password='root',
    database='nhl'
)

cursor = conn.cursor()

In [12]:
create_table_query = """
CREATE TABLE IF NOT EXISTS game_stats (
    stat_id INT PRIMARY KEY,
    game_id INT NOT NULL,
    player_id INT NOT NULL,
    team_id INT NOT NULL,
    goals INT DEFAULT 0,
    assists INT DEFAULT 0,
    points INT DEFAULT 0,
    shots_on_goal INT DEFAULT 0,
    penalty_min INT DEFAULT 0,
    toi VARCHAR(10),
    plus_minus INT DEFAULT 0
)
"""

cursor.execute(create_table_query)

insert_query = """
INSERT IGNORE INTO game_stats (
    stat_id,
    game_id,
    player_id,
    team_id,
    goals,
    assists,
    points,
    shots_on_goal,
    penalty_min,
    toi,
    plus_minus
)
VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
"""

columns = [
    'stat_id',
    'game_id',
    'player_id',
    'team_id',
    'goals',
    'assists',
    'points',
    'shots_on_goal',
    'penalty_min',
    'toi',
    'plus_minus',
]
records = list(
    df[columns]
    .astype(object)
    .where(pd.notna(df[columns]), None)
    .itertuples(index=False, name=None)
)

cursor.executemany(insert_query, records)
conn.commit()
print(f"Inserted {cursor.rowcount} game-stat rows into game_stats")


Inserted 47408 game-stat rows into game_stats
